# Exercise 9: GRPO for Verifiable Reasoning

**Learning goals:**
1. Understand GRPO as a natural fit for **reasoning with verifiable rewards**.
2. Implement the **GRPO loss**: advantage normalisation, PPO clipping, KL regularisation.
3. Understand how **reward design** and **group size** affect the gradient signal.
4. Compare **MAD normalisation** vs std, and the effect of PPO clipping.

## Setup

- This notebook was tested on a Kaggle **GPU: P100 16 GB.**  so we strongly recommend to use the same.
- We will use the `Qwen3-0.6B` model, that peaks at ~8 GB during training. If you the P100 GPU, remember not to keep two full models loaded simultaneously.
- You will sometimes be prompted to restart the kernel, do that to avoid out of memory erros.

Packages: `transformers` + `peft` for the model, `pandas`/`pyarrow` for data, `reasoning_gym` for the generation demo.

In [ ]:
import sys, subprocess

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# --no-deps: install exactly the listed package, touch nothing else.
# Kaggle already has numpy/pandas/etc; we only need to swap the ML stack versions.

# Torch + torchvision — matched cu121 wheels so torchvision is GPU-compatible.
_pip("--force-reinstall", "--no-deps",
     "--index-url", "https://download.pytorch.org/whl/cu121",
     "torch==2.5.1", "torchvision==0.20.1")

# Core ML packages — exact versions from the teaching cluster.
_pip("--force-reinstall", "--no-deps",
     "transformers==4.57.6", "tokenizers==0.22.2",
     "huggingface-hub==0.36.2",
     "accelerate==1.10.1",  "safetensors==0.6.2",
     "peft==0.17.1",        "sentencepiece==0.2.1")

# reasoning-gym — not pre-installed on Kaggle.
_pip("reasoning-gym==0.1.25")

print("Setup complete — restart the kernel before continuing.")

In [ ]:
# torch: tensors & autograd | transformers: pretrained model | peft: LoRA adapters
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True 
import gc
import time
import math
import re
import copy
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## Task: Countdown

The model receives 3 numbers and a target. It must output `\boxed{expression}` that combines those numbers with +, −, ×, ÷ to hit the target exactly.

We use a pre-split `easy_compact` dataset (3 numbers, difficulty=1). Training: 1,200 examples; test: 200. 

**To download the data and upload the on kaggle follow this steps:**

- Download the data (the 2 .parquet files) from https://drive.google.com/drive/folders/1Th0h4aEkMlnH5Iv-lFwB1C6LXji69KBo?usp=share_link
- On the Kaggle Notebook, click "Upload" on the top right part of the screen, then "New Dataset"
- Click "Browse Files" and then select the 2 parquet files
- Enter "countdown" as in the "Dataset Title" box
- Click on "Create"
- You have successfully created the dataset, now ensure to insert your kaggle username in the cell below, so that data will be loaded correctly (if you don't know the username, right click on the dataset folder and copy paste the path)

In [ ]:
COUNTDOWN_DATA_DIR = Path("/kaggle/input/datasets/<YOUR_KAGGLE_USERNAME>/countdown")
train_df = pd.read_parquet(COUNTDOWN_DATA_DIR / "train_countdown_easy_compact.parquet")
test_df  = pd.read_parquet(COUNTDOWN_DATA_DIR / "test_countdown_easy_compact.parquet")
print(f"Train: {len(train_df)} rows  |  Test: {len(test_df)} rows")

# Countdown: given a set of numbers, find an arithmetic expression that hits the target.
row = train_df.iloc[0]
print(f"\nExample — numbers: {row['numbers']}, target: {row['target']}")
print(row["prompt"][:600])

## Reasoning Gym

`reasoning_gym` can generate Countdown problems programmatically — useful for scaling to larger datasets or harder difficulties. Here we just show 3 examples to illustrate the task format.

In [ ]:
# On-the-fly generation with Reasoning Gym
try:
    import reasoning_gym

    rg_countdown = reasoning_gym.create_dataset(
        "countdown",
        size=3,
        seed=SEED,
        min_numbers=3,
        max_numbers=3,
        min_target=20,
        max_target=120,
    )

    print("Generated Countdown items from Reasoning Gym")
    print("=" * 90)
    for i, item in enumerate(rg_countdown):
        print(f"Item {i}")
        print("Prompt:", item.get("question") or item.get("prompt"))
        print("Answer:", item.get("answer"))
        print("Metadata:", item.get("metadata"))
        print("-" * 90)
except Exception as e:
    print("Reasoning Gym generation demo is unavailable in this environment:")
    print(type(e).__name__, e)

## Model

We use `Qwen3-0.6B` — a 600 M parameter instruction-tuned model with a built-in thinking mode. We disable thinking (`/no_think`) so the model outputs a direct answer in ~14 tokens, keeping training fast on a P100.

In [ ]:
MODEL_SPEC = {
    "hf_id":          "Qwen/Qwen3-0.6B",
    "label":          "Qwen3-0.6B",
    "max_new_tokens": 96,    # short enough for \boxed{expr}; keeps training fast
    "temperature":    0.7,   # sampling diversity for generating G completions per prompt
    "top_p":          0.8,
    "top_k":          20,
}
PROMPT_LEN     = 128
MAX_NEW_TOKENS = MODEL_SPEC["max_new_tokens"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_SPEC["hf_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # prompts left-padded so all responses start at the same position
print(f"Loaded tokenizer: {MODEL_SPEC['label']}")

## Examples

Convert each dataframe row into a plain dict and draw 1,200 training examples and 200 test examples.

In [ ]:
def row_to_example(row) -> Dict:
    return {
        "prompt": row["prompt"],
        "numbers": [int(x) for x in row["numbers"]],
        "target": int(row["target"]),
        "difficulty": int(row["difficulty"]),
        "source_question": row["source_question"],
    }


train_examples = [row_to_example(row) for _, row in train_df.iloc[:1200].iterrows()]
test_examples  = [row_to_example(row) for _, row in test_df.iloc[:200].iterrows()]

print("First training example:")
print(train_examples[0])

## Prompt & Generation Helpers

These handle the full lifecycle of one GRPO iteration step:
- **`format_prompt`** — applies the Qwen3 chat template with `/no_think`
- **`encode_prompts`** — tokenises and left-pads to a fixed `PROMPT_LEN` so all responses start at the same position
- **`generate_responses`** — runs `model.generate`; stochastic during training (diverse G samples), greedy during eval
- **`get_response_log_probs`** — extracts per-token log probs **for response tokens only** — required for the GRPO loss ratio

In [ ]:
def format_prompt(text: str) -> str:
    # /no_think suppresses Qwen3's built-in chain-of-thought; model outputs answer directly.
    msgs = [{"role": "user", "content": "/no_think\n" + text}]
    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=True,
                                              enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False,
                                              add_generation_prompt=True)

def encode_prompts(prompts: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
    formatted = [format_prompt(p) for p in prompts]
    enc = tokenizer(formatted, return_tensors="pt", padding="max_length",
                    truncation=True, max_length=PROMPT_LEN).to(device)
    return enc.input_ids, enc.attention_mask

@torch.no_grad()
def generate_responses(model, p_ids, p_mask, greedy=False, max_new_tokens=None):
    kwargs = dict(input_ids=p_ids, attention_mask=p_mask,
                  max_new_tokens=max_new_tokens or MAX_NEW_TOKENS,
                  min_new_tokens=8, pad_token_id=tokenizer.pad_token_id)
    if greedy:         # greedy=True for eval (deterministic); False for training (diverse samples)
        kwargs["do_sample"] = False
    else:
        kwargs.update(do_sample=True, temperature=MODEL_SPEC["temperature"],
                      top_p=MODEL_SPEC["top_p"], top_k=MODEL_SPEC["top_k"])
    return model.generate(**kwargs)

def make_full_mask(p_mask, B, full_ids):
    gen_len = full_ids.shape[1] - p_mask.shape[1]
    return torch.cat([p_mask, torch.ones(B, gen_len, device=device, dtype=torch.long)], dim=1)

def decode_responses_only(full_ids, p_ids):
    return tokenizer.batch_decode(full_ids[:, p_ids.shape[1]:], skip_special_tokens=True)

def get_response_log_probs(lm, input_ids, attention_mask):
    # Returns per-token log probs masked to response tokens only (prompt tokens zeroed out).
    logits = lm(input_ids=input_ids, attention_mask=attention_mask).logits
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    token_lps = log_probs.gather(2, input_ids[:, 1:].unsqueeze(2)).squeeze(2)
    prompt_lengths = attention_mask[:, :PROMPT_LEN].sum(dim=1)
    mask = torch.zeros_like(token_lps)
    for i, plen in enumerate(prompt_lengths.tolist()):
        start = max(int(plen) - 1, 0)
        end = attention_mask.shape[1] - 1
        mask[i, start:end] = attention_mask[i, 1:][start:end].float()
    per_token = token_lps * mask
    return per_token.sum(dim=-1), per_token, mask

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Verifiable Reward for Countdown

The verifier checks three things: did the model output a `\boxed{}`, is the expression valid arithmetic, and does it use the right numbers and hit the target?

Partial credit: `0.1` for a valid expression with the right numbers, `1.0` for hitting the target. This prevents full advantage collapse when solve rate is low.

In [ ]:
BOX_RE = re.compile(r"\\boxed\{([^}]*)\}")
NUMBER_RE = re.compile(r"\d+")


def extract_boxed_answer(text: str) -> str | None:
    matches = BOX_RE.findall(text)
    return matches[-1].strip() if matches else None


def safe_eval_expr(expr: str) -> float:
    return float(eval(expr, {"__builtins__": {}}))


def verify_countdown_answer(response_text: str, numbers: List[int], target: int) -> Dict[str, float]:
    boxed = extract_boxed_answer(response_text)
    info = {
        "has_box": 0.0,
        "valid_expr": 0.0,
        "uses_right_numbers": 0.0,
        "hits_target": 0.0,
        "reward": 0.0,
    }
    if boxed is None:
        return info

    info["has_box"] = 1.0

    try:
        value = safe_eval_expr(boxed)
        info["valid_expr"] = 1.0
    except Exception:
        return info

    used_numbers = sorted(int(x) for x in NUMBER_RE.findall(boxed))
    gold_numbers = sorted(numbers)
    if used_numbers == gold_numbers:
        info["uses_right_numbers"] = 1.0

    if info["uses_right_numbers"] and math.isfinite(value) and abs(value - target) < 1e-6:
        info["hits_target"] = 1.0

    # -- TODO: assign info["reward"] ------------------------------------------
    # valid expr with right numbers → partial credit 0.1
    # hits_target → full credit 1.0
    info["reward"] = ...  # YOUR CODE HERE
    # -------------------------------------------------------------------------

    return info


def compute_countdown_rewards(texts: List[str], batch_examples: List[Dict]) -> Tuple[torch.Tensor, List[Dict[str, float]]]:
    infos = [
        verify_countdown_answer(text, ex["numbers"], ex["target"])
        for text, ex in zip(texts, batch_examples)
    ]
    rewards = torch.tensor([x["reward"] for x in infos], device=device, dtype=torch.float32)
    return rewards, infos

In [ ]:
# Verifier sanity check
sample_ex = train_examples[0]

cases = [
    ("\\boxed{5+17+91}",    "valid expr + right numbers + hits target → reward=1.0"),
    ("\\boxed{91-17+5}",    "valid expr + right numbers but wrong target → reward=0.1"),
    ("\\boxed{5+17}",       "valid expr but wrong numbers → reward=0.0"),
]
for response, desc in cases:
    info = verify_countdown_answer(response, sample_ex["numbers"], sample_ex["target"])
    print(f"{desc}")
    print(f"  → reward={info['reward']}")

## GRPO Policy

We attach a LoRA adapter to the pretrained model. The base weights stay frozen and act as the reference policy for the KL penalty — so the trained policy is penalised for drifting too far from where it started.

We skip a separate SFT warmup and go straight from pretrained weights. In a real pipeline you would typically warm-start from an SFT checkpoint first.

In [ ]:
def create_rl_model():
    m = AutoModelForCausalLM.from_pretrained(MODEL_SPEC["hf_id"]).to(device)
    m.config.pad_token_id = tokenizer.pad_token_id
    # LoRA injects small trainable matrices into attention + MLP projections (~1% of params).
    # The frozen base weights serve double duty as the reference policy for the KL penalty.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    lora_cfg = LoraConfig(
        r=16,          # rank — higher = more capacity but more memory
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    m = get_peft_model(m, lora_cfg)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    print(f"LoRA trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return m

## GRPO Loss

Three components:
1. **Advantage normalisation**: within-group mean/std — the group mean is the baseline, no critic needed.
2. **PPO clipping**: ratio clipped to `[1−ε, 1+ε]` — prevents destabilising large updates. `clip_eps=0` disables clipping (vanilla policy gradient).
3. **KL penalty**: k3 estimator (always ≥ 0) keeps the policy close to the frozen reference.

In [ ]:
def grpo_loss(grpo_model, full_ids, full_mask, old_per_token_lps, rewards,
              group_size, beta=0.01, clip_eps=0.2):
    B = full_ids.shape[0]
    n_prompts = B // group_size
    r_grouped = rewards.view(n_prompts, group_size)

    # TODO 1: Within each group of G responses, subtract the group mean and divide
    # by the group std to get zero-mean, unit-variance advantages. Shape: [B].
    raise NotImplementedError("TODO 1: compute g_mean, g_std, advantages")

    _, curr_pt, resp_mask = get_response_log_probs(grpo_model, full_ids, full_mask)
    ratio = torch.exp(curr_pt - old_per_token_lps.detach())  # π_new / π_old per token

    # TODO 2: Weight advantages to per-token level (masked to response tokens).
    # Compute the unclipped objective (surr1) and the ratio-clipped objective (surr2),
    # then take the conservative minimum. If clip_eps=0, no clipping — use surr1 directly.
    raise NotImplementedError("TODO 2: compute token_adv, surr1, surr2, clipped")

    with torch.no_grad():
        with grpo_model.disable_adapter():
            _, ref_pt, _ = get_response_log_probs(grpo_model, full_ids, full_mask)

    # k3 KL estimator — always non-negative.
    log_ratio = ref_pt - curr_pt
    pt_kl = (torch.exp(log_ratio) - log_ratio - 1) * resp_mask

    # TODO 3: Average the per-token loss within each sequence (not across all tokens),
    # then average across sequences. Negate because we minimise loss but maximise reward.
    raise NotImplementedError("TODO 3: compute seq_lens, seq_loss, loss")

    with torch.no_grad():
        seq_kl    = pt_kl.detach().sum(dim=-1) / resp_mask.sum(dim=-1).clamp(min=1)
        clip_mask = ((ratio - 1.0).abs() > clip_eps) & (resp_mask > 0)
    return loss, {
        "reward":    rewards.mean().item(),
        "kl":        seq_kl.mean().item(),
        "clip_frac": clip_mask.sum().item() / resp_mask.sum().item(),
        "adv_std":   advantages.std().item(),
    }

## Training Loop

`train_grpo` accepts an optional `loss_fn` argument (defaults to `grpo_loss`) so we can swap in alternative loss functions for exercises.

In [ ]:
def sample_prompt_batch(examples, n_prompts, group_size):
    idx = np.random.choice(len(examples), n_prompts, replace=False)
    batch = [examples[i] for i in idx for _ in range(group_size)]
    return batch, [ex["prompt"] for ex in batch]


def evaluate_model(model, examples, n=200, label="Model"):
    subset, all_infos, all_texts = examples[:n], [], []
    model.eval()
    for i in range(0, len(subset), 32):
        batch = subset[i:i + 32]
        p_ids, p_mask = encode_prompts([ex["prompt"] for ex in batch])
        with torch.no_grad():
            full_ids = generate_responses(model, p_ids, p_mask, greedy=True)
        texts = decode_responses_only(full_ids, p_ids)
        _, infos = compute_countdown_rewards(texts, batch)
        all_texts.extend(texts)
        all_infos.extend(infos)
    solve  = np.mean([x["hits_target"] for x in all_infos])
    reward = np.mean([x["reward"] for x in all_infos])
    print(f"{label}: solve={solve:.1%}  reward={reward:.3f}")
    return {"solve_rate": solve, "mean_reward": reward}, all_texts, all_infos


def train_grpo(n_iters=60, group_size=4, grpo_epochs=2,
               lr=5e-6, beta=0.001, clip_eps=0.2, loss_fn=None):
    if loss_fn is None:
        loss_fn = grpo_loss
    model = create_rl_model()
    opt   = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr)
    hist  = {"reward": [], "kl": [], "clip_frac": [], "solve_rate": [], "adv_std": []}
    t0    = time.time()
    for it in range(n_iters):
        batch, prompts = sample_prompt_batch(train_examples, 1, group_size)
        p_ids, p_mask  = encode_prompts(prompts)
        # Phase 1 — Generate: sample G completions stochastically (no gradients).
        model.eval()
        full_ids  = generate_responses(model, p_ids, p_mask)
        full_mask = make_full_mask(p_mask, group_size, full_ids)
        texts     = decode_responses_only(full_ids, p_ids)
        # Phase 2 — Score: run verifier to get rewards; snapshot log probs before the update.
        rewards, reward_infos = compute_countdown_rewards(texts, batch)
        with torch.no_grad():
            _, old_lps, _ = get_response_log_probs(model, full_ids, full_mask)
        # Phase 3 — Update: GRPO loss + gradient step, repeated grpo_epochs times.
        model.train()
        for _ in range(grpo_epochs):
            loss, info = loss_fn(model, full_ids, full_mask, old_lps, rewards,
                                 group_size=group_size, beta=beta, clip_eps=clip_eps)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        for k in ["reward", "kl", "clip_frac", "adv_std"]:
            hist[k].append(info[k])
        hist["solve_rate"].append(float(np.mean([x["hits_target"] for x in reward_infos])))
        if (it + 1) % 15 == 0:
            print(f"iter {it+1:3d}/{n_iters} | reward {info['reward']:.3f} | "
                  f"solve {hist['solve_rate'][-1]:.1%} | KL {info['kl']:.4f} "
                  f"| clip {info['clip_frac']:.0%} | {time.time()-t0:.0f}s")
    return hist, model

In [ ]:
# Baseline — fresh model, greedy eval on 200 test examples
cleanup_memory()
_tmp = AutoModelForCausalLM.from_pretrained(MODEL_SPEC["hf_id"]).to(device)
_tmp.config.pad_token_id = tokenizer.pad_token_id
base_results, _, _ = evaluate_model(_tmp, test_examples, n=200, label="Qwen3-0.6B base")
del _tmp; cleanup_memory()

# Train
grpo_history, grpo_model = train_grpo()

# Post-training eval
grpo_results, _, _ = evaluate_model(grpo_model, test_examples, n=200, label="Qwen3-0.6B + GRPO")
print(f"\nSolve rate gain: +{(grpo_results['solve_rate'] - base_results['solve_rate']):.1%} over 60 iters")

**What you should see:** Reward and solve rate trend upward over 60 iterations. KL stays small (< 0.02) — the policy improves without diverging from the pretrained weights. The final jump from ~20 % → ~30 % solve rate shows that verifiable reward alone is enough to teach arithmetic formatting. Training-time solve rate is noisy (4 samples per iter); the greedy eval on 200 examples is the reliable signal.

In [ ]:
def plot_history(history, label="GRPO"):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (k, title, color) in zip(axes, [
        ("reward",     "Mean Reward",       "seagreen"),
        ("solve_rate", "Solve Rate",        "steelblue"),
        ("kl",         "KL from Reference", "coral"),
    ]):
        data = np.array(history[k])
        ax.plot(data, alpha=0.3, color=color)
        w = 8
        if len(data) >= w:
            sm = np.convolve(data, np.ones(w) / w, mode="valid")
            ax.plot(range(w - 1, len(data)), sm, color=color, lw=2)
        ax.set_title(title)
        ax.set_xlabel("Iteration")
    plt.suptitle(label)
    plt.tight_layout()
    plt.show()

# Reward and solve rate should trend up; KL should stay small (< 0.02).
plot_history(grpo_history)

In [ ]:
# Look at whether the model reasons step-by-step or jumps straight to \boxed{...}.
for ex in test_examples[:3]:
    p_ids, p_mask = encode_prompts([ex["prompt"]])
    grpo_model.eval()
    full_ids = generate_responses(grpo_model, p_ids, p_mask)
    text = decode_responses_only(full_ids, p_ids)[0]
    info = verify_countdown_answer(text, ex["numbers"], ex["target"])
    print("=" * 90)
    print("Prompt numbers:", ex["numbers"], "target:", ex["target"])
    print(text[:600])
    print("Verifier:", info)

**What you should notice:** The model almost always jumps straight to `\\boxed{expression}` — no reasoning steps. GRPO optimised for reward, not explanation. We'll revisit this in Exercise 6.

## Exercises

**Reward and group structure**
1. Binary vs shaped reward
2. Group size sweep

**Algorithmic variants**

3. MAD normalisation
4. Effect of clipping

**Exploration**

5. Generation temperature and diversity

**Reasoning analysis**

6. Does the trained model reason, or just shortcut?

---
### Exercise 1: Binary vs Shaped Reward

The current verifier gives `0.1` for a valid boxed expression with the right numbers and `1.0` for a correct solve. A strict binary reward gives only `0.0` or `1.0`.

Tradeoff:
- **shaped**: more non-zero rewards → less advantage collapse, but may reinforce format over correctness.
- **binary**: cleaner signal, but more collapsed groups (all zeros) → weaker gradients early on.

Run the cell below on the trained model to compare reward variance statistics for both modes.

In [ ]:
def compute_countdown_rewards_configurable(
    texts: List[str], batch_examples: List[Dict], mode: str = "shaped"
) -> Tuple[torch.Tensor, List[Dict[str, float]]]:
    # mode='shaped': 0.1 for valid boxed expr with right numbers, 1.0 for correct solve.
    # mode='binary': 0.0 or 1.0 only.

    # -- TODO: implement both reward modes ------------------------------------
    # 1. Run verify_countdown_answer on all (text, example) pairs.
    # 2. mode='binary': set info["reward"] = float(info["hits_target"]) for each.
    #    mode='shaped': leave rewards as set by verify_countdown_answer.
    # 3. Build and return the rewards tensor and infos list.
    raise NotImplementedError
    # -------------------------------------------------------------------------


@torch.no_grad()
def compare_reward_modes(model, n_prompts=8, group_size=4):
    batch_examples, prompt_texts = sample_prompt_batch(train_examples, n_prompts, group_size)
    p_ids, p_mask = encode_prompts(prompt_texts)
    full_ids = generate_responses(model, p_ids, p_mask)
    texts = decode_responses_only(full_ids, p_ids)

    rows = []
    for mode in ("shaped", "binary"):
        rewards, _ = compute_countdown_rewards_configurable(texts, batch_examples, mode=mode)
        r_grouped = rewards.view(n_prompts, group_size)
        stds = r_grouped.std(dim=1)
        rows.append({
            "mode": mode,
            "mean_reward": rewards.mean().item(),
            "collapsed_pct": (stds < 1e-3).float().mean().item(),
        })
    df = pd.DataFrame(rows)
    display(df)
    return df


_base_ex1 = AutoModelForCausalLM.from_pretrained(MODEL_SPEC["hf_id"]).to(device)
_base_ex1.config.pad_token_id = tokenizer.pad_token_id
reward_mode_df = compare_reward_modes(_base_ex1, n_prompts=32)
del _base_ex1; cleanup_memory()

**What to expect:** `collapsed_pct` is the key metric — shaped reward has significantly lower collapsed fraction than binary (roughly half as many dead groups). Binary collapses whenever all completions in a group are wrong (all 0.0). Shaped keeps more groups alive by giving 0.1 partial credit for valid expressions with the right numbers, so a group only collapses when every completion is either invalid or uses wrong numbers.

---
### Exercise 2: Group Size Sweep

Larger groups give GRPO more samples to normalise rewards over, reducing advantage collapse — but each iteration uses more GPU memory and time.

Run the cell below on the trained model and observe:
- does reward variance within groups increase with `group_size`?
- does the collapsed-group fraction drop?

In [ ]:
@torch.no_grad()
def sweep_group_sizes(model, group_sizes=(2, 4, 8), n_prompts=8):
    rows = []
    for gs in group_sizes:
        batch_examples, prompt_texts = sample_prompt_batch(train_examples, n_prompts, gs)
        p_ids, p_mask = encode_prompts(prompt_texts)
        full_ids = generate_responses(model, p_ids, p_mask)
        texts = decode_responses_only(full_ids, p_ids)
        rewards, _ = compute_countdown_rewards(texts, batch_examples)
        r_grouped = rewards.view(n_prompts, gs)
        stds = r_grouped.std(dim=1)
        rows.append({
            "group_size": gs,
            "mean_reward": rewards.mean().item(),
            "mean_group_std": stds.mean().item(),
            "collapsed_pct": (stds < 1e-3).float().mean().item(),
        })
    df = pd.DataFrame(rows)
    display(df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar([str(gs) for gs in df["group_size"]], df["mean_group_std"])
    axes[0].set_xlabel("group_size")
    axes[0].set_ylabel("Mean within-group reward std")
    axes[0].set_title("Reward variance vs group_size")

    axes[1].bar([str(gs) for gs in df["group_size"]], df["collapsed_pct"])
    axes[1].set_xlabel("group_size")
    axes[1].set_ylabel("Collapsed group fraction (std < 1e-3)")
    axes[1].set_title("Advantage collapse vs group_size")

    plt.tight_layout()
    plt.show()
    return df


_base_ex2 = AutoModelForCausalLM.from_pretrained(MODEL_SPEC["hf_id"]).to(device)
_base_ex2.config.pad_token_id = tokenizer.pad_token_id
group_size_sweep_df = sweep_group_sizes(_base_ex2, n_prompts=8)
del _base_ex2; cleanup_memory()

**What to expect:** Larger groups → higher within-group std → fewer collapsed groups. With G=2 you often get two identical outputs (both right or both wrong) → collapsed advantage. With G=8 the group is more likely to contain a mix → non-zero gradient. Cost: training time and GPU memory scale linearly with G.

---
### Exercise 3: MAD Normalisation

Standard GRPO normalises within-group advantages by the standard deviation:

```
A_i = (r_i - mean) / std
```

For binary rewards (0/1), the update magnitude is proportional to `2G*sqrt(p*(1-p))`, which peaks at p=0.5 and shrinks toward zero for easy (p->1) and hard (p->0) prompts. The model gets its weakest gradient signal on the problems it finds hardest or easiest.

**A simple fix (Harder Is Better, 2025):** normalise by the Mean Absolute Deviation (MAD) instead:

```
A_i = (r_i - mean) / MAD,   where MAD = (1/G) * sum(|r_i - mean|)
```

This produces a constant update magnitude of G regardless of difficulty -- hard and easy prompts contribute equally.

**Your task:**
1. Implement `grpo_loss_mad` -- identical to `grpo_loss` except replace std normalisation with MAD.
2. Run both for 60 iterations and compare training curves.

**💭 Question:** Do you expect MAD converge faster, slower, or similarly on Countdown? Why might the difference be small here compared to a harder task?

**Answer**:

In [ ]:
def grpo_loss_mad(grpo_model, full_ids, full_mask, old_per_token_lps, rewards,
                  group_size, beta=0.001, clip_eps=0.2):
    B, n_prompts = full_ids.shape[0], full_ids.shape[0] // group_size
    r_grouped = rewards.view(n_prompts, group_size)

    # TODO 1: Compute within-group mean, MAD, and normalised advantages.
    # MAD = mean of |r_i - mean| within each group — replaces std from grpo_loss.
    g_mean     = ...   # YOUR CODE HERE
    mad        = ...   # YOUR CODE HERE
    advantages = ...   # YOUR CODE HERE

    _, curr_pt, resp_mask = get_response_log_probs(grpo_model, full_ids, full_mask)
    ratio     = torch.exp(curr_pt - old_per_token_lps.detach())
    token_adv = advantages.detach().unsqueeze(1) * resp_mask
    surr1     = ratio * token_adv
    surr2     = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * token_adv
    clipped   = torch.min(surr1, surr2)

    with torch.no_grad():
        with grpo_model.disable_adapter():
            _, ref_pt, _ = get_response_log_probs(grpo_model, full_ids, full_mask)
    log_ratio = ref_pt - curr_pt
    pt_kl     = (torch.exp(log_ratio) - log_ratio - 1) * resp_mask

    # TODO 2: Sequence-level loss aggregation — same as grpo_loss TODO 3.
    seq_lens = ...   # YOUR CODE HERE
    seq_loss = ...   # YOUR CODE HERE
    loss     = ...   # YOUR CODE HERE

    with torch.no_grad():
        seq_kl    = pt_kl.detach().sum(dim=-1) / resp_mask.sum(dim=-1).clamp(min=1)
        clip_mask = ((ratio - 1.0).abs() > clip_eps) & (resp_mask > 0)
    return loss, {
        "reward":    rewards.mean().item(),
        "kl":        seq_kl.mean().item(),
        "clip_frac": clip_mask.sum().item() / resp_mask.sum().item(),
        "adv_std":   advantages.std().item(),
    }


history_std = grpo_history  # reuse main run as std baseline

print("Training with MAD normalisation...")
cleanup_memory()
history_mad, model_mad = train_grpo(loss_fn=grpo_loss_mad)
eval_mad, _, _ = evaluate_model(model_mad, test_examples, n=200, label="GRPO+MAD")
del model_mad; cleanup_memory()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ["reward", "solve_rate"], ["Mean Reward", "Solve Rate"]):
    for hist, lbl, col in [(history_std, "std", "steelblue"), (history_mad, "MAD", "coral")]:
        data = np.array(hist[key])
        ax.plot(data, alpha=0.3, color=col)
        w = 8
        if len(data) >= w:
            ax.plot(range(w-1, len(data)), np.convolve(data, np.ones(w)/w, mode="valid"),
                    color=col, lw=2, label=lbl)
    ax.set_title(title); ax.set_xlabel("Iteration"); ax.legend()
plt.suptitle("std vs MAD normalisation"); plt.tight_layout(); plt.show()

**What to expect:** MAD and std converge to similar final solve rates on Countdown's easy split. The difference is larger on harder tasks where many groups have near-zero std (all failures) — MAD gives those groups a non-zero gradient signal anyway. On this task the base solve rate is ~20 %, so std rarely collapses completely.

---
### Exercise 4: Effect of the Clipping Mechanism

The clip in `grpo_loss` implements a PPO-style trust region: when the probability ratio leaves `[1−ε, 1+ε]`, the gradient is cut off. Setting `clip_eps=0` disables clipping entirely — the loss becomes plain policy gradient with no trust region.

**Your task:** Run training with `clip_eps=0.2` (standard) and `clip_eps=0` (no clip — vanilla PG) and compare training curves and final solve rate. The `clip_eps=0.2` run reuses the history from the main training run.


**💭 Question:** Does removing clipping cause instability here? Think about *when* clipping matters most and why it might be less critical at small scale and low learning rate.

**Answer:**

In [ ]:
history_clipped = grpo_history  # reuse main run

print("Running clip_eps=0 (vanilla PG, no trust region)...")
cleanup_memory()
history_noclip, model_noclip = train_grpo(clip_eps=0.0)
eval_noclip, _, _ = evaluate_model(model_noclip, test_examples, n=200, label="GRPO clip_eps=0")
del model_noclip; cleanup_memory()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (key, title) in zip(axes, [
    ("reward",    "Mean Reward"),
    ("solve_rate","Solve Rate"),
    ("adv_std",   "Advantage Std"),
]):
    for hist, lbl, col in [
        (history_clipped, "clip_eps=0.2", "steelblue"),
        (history_noclip,  "clip_eps=0 (vanilla PG)", "coral"),
    ]:
        data = np.array(hist[key])
        ax.plot(data, alpha=0.3, color=col)
        w = 8
        if len(data) >= w:
            ax.plot(range(w-1, len(data)), np.convolve(data, np.ones(w)/w, mode="valid"),
                    color=col, lw=2, label=lbl)
    ax.set_title(title); ax.set_xlabel("Iteration"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
comparison = pd.DataFrame([
    {"method": "GRPO (std, clip=0.2)", "solve_rate": f"{grpo_results['solve_rate']:.1%}", "mean_reward": f"{grpo_results['mean_reward']:.3f}"},
    {"method": "MAD normalisation",    "solve_rate": f"{eval_mad['solve_rate']:.1%}",     "mean_reward": f"{eval_mad['mean_reward']:.3f}"},
    {"method": "No clip (vanilla PG)", "solve_rate": f"{eval_noclip['solve_rate']:.1%}",  "mean_reward": f"{eval_noclip['mean_reward']:.3f}"},
])
display(comparison)

**Key result:** Standard GRPO (std + clip) and MAD normalisation perform similarly at 60 iterations. Removing the clip (`clip_eps=0`) hurts: without a trust region the policy can take large steps on noisy batches and destabilise. The effect is more dramatic at higher learning rates or longer runs.

---
### Exercise 5: Generation Temperature and Within-Group Diversity

GRPO generates G completions per prompt using temperature sampling. Temperature controls how diverse those completions are -- and diversity is essential: if all G completions are identical, within-group reward variance is zero and the gradient vanishes.

**Think about the extremes before running:**
- `temperature -> 0` (near-greedy): all completions collapse to the same output -> zero within-group variance -> no learning signal
- `temperature -> inf`: completions are random noise -> all wrong -> zero variance again
- There is a Goldilocks range where some completions succeed and some fail -> non-zero advantage

**Your task:**
1. Run `inspect_group_rewards` on a fresh base model at temperatures [0.1, 0.4, 0.7, 1.2].
2. Record mean group std and collapsed fraction at each temperature.
3. Based on the results, explain why we use temperature=0.7.

In [ ]:
temp_results = []
# Sample once so all temperatures see the same prompts — controls for prompt difficulty.
_temp_batch, _temp_prompts = sample_prompt_batch(train_examples, 32, 4)
p_ids, p_mask = encode_prompts(_temp_prompts)

_base_temp = AutoModelForCausalLM.from_pretrained(MODEL_SPEC["hf_id"]).to(device)
_base_temp.config.pad_token_id = tokenizer.pad_token_id

for temp in [0.1, 0.4, 0.7, 1.2]:
    MODEL_SPEC["temperature"] = temp
    with torch.no_grad():
        full_ids = generate_responses(_base_temp, p_ids, p_mask)
        texts = decode_responses_only(full_ids, p_ids)
        rewards, _ = compute_countdown_rewards(texts, _temp_batch)

    # YOUR CODE HERE: reshape rewards into groups, compute per-group std, and record stats.
    # r_grouped shape: [n_prompts, group_size] — each row is one prompt's G rewards.
    # collapsed_pct: fraction of groups where within-group std < 1e-3 (all rewards identical).
    r_grouped = ...
    stds      = ...
    temp_results.append({
        "temperature":    temp,
        "mean_group_std": ...,
        "collapsed_pct":  ...,
        "mean_reward":    ...,
    })
    print(f"temp={temp:.1f} | mean_group_std={stds.mean().item():.4f} | "
          f"collapsed={(stds < 1e-3).float().mean().item():.0%} | mean_reward={rewards.mean().item():.3f}")

MODEL_SPEC["temperature"] = 0.7
del _base_temp; cleanup_memory()

df_temp = pd.DataFrame(temp_results)
display(df_temp)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df_temp["temperature"], df_temp["mean_group_std"], marker="o", color="steelblue")
axes[0].set_xlabel("Temperature")
axes[0].set_ylabel("Mean within-group reward std")
axes[0].set_title("Reward variance vs sampling temperature")
axes[1].plot(df_temp["temperature"], df_temp["collapsed_pct"], marker="o", color="coral")
axes[1].set_xlabel("Temperature")
axes[1].set_ylabel("Collapsed group fraction (std < 1e-3)")
axes[1].set_title("Advantage collapse vs sampling temperature")
plt.tight_layout()
plt.show()

**What to expect:** At low temperature (~0.1) nearly all completions are identical → variance ≈ 0 → collapsed groups ≈ 100% → no learning signal. At high temperature (~1.2) diversity increases but correctness drops. Temperature = 0.7 is the Goldilocks zone: enough diversity for non-zero within-group variance, high enough quality for some correct answers to appear in each group.

---
### Exercise 6: Does the trained model reason, or just shortcut?

After GRPO training the model's solve rate jumps from 19.5 % → 31 %, but look at the raw outputs: they are almost always a single line like `\boxed{96 + 73 + 8}` — no reasoning whatsoever. The reward function only checks correctness; it never required the model to explain its steps, so the model learned to skip straight to the answer.

This matters because:
- A model that reasons generalises better (harder problems, different tasks).
- Format-optimised shortcuts score well on the training distribution but fail in the wild.

**Part A** — Analyse output structure on the trained model: response length and fraction with visible reasoning.

**Part B** — *Inference only* on a fresh base model with Qwen3's built-in thinking mode (`/think` system prompt, `max_new_tokens=512`). Measure the solve rate vs the no-think baseline.

**Part C** — Why can't we simply train with thinking enabled?
On a P100 (16 GB), peak activation memory at `max_new_tokens=96` is ~6 GB. With `max_new_tokens=512` the sequence length is 5× longer; attention activations scale as O(n²), so the training footprint roughly exceeds 16 GB → OOM. This is an active research challenge: efficient long-CoT RL training.

In [ ]:
import re

def analyse_response_structure(model, examples, n=30, greedy=True):
    # Sample n responses and measure length / reasoning presence.
    subset = examples[:n]
    lengths, has_reasoning, texts = [], [], []
    model.eval()
    for ex in subset:
        p_ids, p_mask = encode_prompts([ex["prompt"]])
        with torch.no_grad():
            full_ids = generate_responses(model, p_ids, p_mask, greedy=greedy)
        text = decode_responses_only(full_ids, p_ids)[0]
        texts.append(text)
        lengths.append(len(tokenizer.encode(text)))
        pre_box = text.split(r'\boxed')[0] if r'\boxed' in text else text
        has_reasoning.append(bool(re.search(r'\d+\s*[+\-*/]\s*\d+', pre_box)))
    print(f"Mean response length : {sum(lengths)/len(lengths):.1f} tokens")
    print(f"Fraction with visible reasoning: {sum(has_reasoning)/len(has_reasoning):.1%}")
    print()
    print("Sample outputs (first 3):")
    for t in texts[:3]:
        print("  " + t[:300].replace("\n", "\n  "))
        print()
    return texts, lengths, has_reasoning

print("=== Trained GRPO model ===")
_, trained_lengths, trained_reasoning = analyse_response_structure(grpo_model, test_examples, n=30)

base_model_tmp = AutoModelForCausalLM.from_pretrained(MODEL_SPEC["hf_id"]).to(device)
base_model_tmp.config.pad_token_id = tokenizer.pad_token_id
print("=== Base model ===")
_, base_lengths, base_reasoning = analyse_response_structure(base_model_tmp, test_examples, n=30)
del base_model_tmp; cleanup_memory()

print(f"Length delta (trained - base): {sum(trained_lengths)/len(trained_lengths) - sum(base_lengths)/len(base_lengths):+.1f} tokens")

**What to expect:** Both base and GRPO models produce very short responses (~14 tokens) with near-zero visible reasoning. GRPO training changed *which expressions* the model guesses, not *whether it reasons*. The reward never required explanation, so the model learned to skip it.

In [ ]:
# Part B: Inference-only evaluation — base model with Qwen3 thinking mode enabled.
# We compare: base (no_think, 96 tokens) vs base (think, 512 tokens).
# No training — just greedy decoding on 50 test examples each.

def encode_prompts_think(prompt_texts):
    # YOUR CODE HERE: encode each prompt with a "/think" system message (not "/no_think").
    # Unlike encode_prompts, prompts vary in length after applying the chat template,
    # so tokenise each one separately, then left-pad all to the batch max length.
    # Return (p_ids, p_mask) tensors on device, same interface as encode_prompts.
    raise NotImplementedError

@torch.no_grad()
def evaluate_model_think(eval_model, eval_examples, n=20, max_new_tokens=1024, label="Model"):
    subset = eval_examples[:n]
    all_infos, tok_counts = [], []
    eval_model.eval()
    for ex in subset:
        p_ids, p_mask = encode_prompts_think([ex["prompt"]])
        full_ids = generate_responses(eval_model, p_ids, p_mask,
                                      greedy=True, max_new_tokens=max_new_tokens)
        text = decode_responses_only(full_ids, p_ids)[0]
        all_infos.append(verify_countdown_answer(text, ex["numbers"], ex["target"]))
        tok_counts.append(len(tokenizer.encode(text)))
    solve = sum(x["hits_target"] for x in all_infos) / len(all_infos)
    boxed = sum(x["has_box"]      for x in all_infos) / len(all_infos)
    avg_tok = sum(tok_counts) / len(tok_counts)
    print(f"{label} | n={n} | solve rate: {solve:.1%} | boxed rate: {boxed:.1%} | avg tokens: {avg_tok:.0f}")
    return {"solve_rate": solve, "boxed_rate": boxed, "avg_tokens": avg_tok}

think_model = AutoModelForCausalLM.from_pretrained(MODEL_SPEC["hf_id"]).to(device)
think_model.config.pad_token_id = tokenizer.pad_token_id

print("Inference comparison — base model only, greedy, n=20")
print("=" * 60)
no_think_res, _, _ = evaluate_model(think_model, test_examples, n=20, label="Base no_think (96 tok)")
think_res    = evaluate_model_think(think_model, test_examples, n=20,
                                    max_new_tokens=1024, label="Base /think (1024 tok)")
del think_model; cleanup_memory()

print()
print("Solve rate delta (think - no_think):",
      f"{think_res['solve_rate'] - no_think_res['solve_rate']:+.1%}")

**Key finding:** The untrained base model with `/think` enabled matches or beats the GRPO-trained model in no-think mode — using zero training, just longer inference. This reveals the core tension: GRPO no-think training is fast and feasible on a P100 (96 tokens), but the model shortcuts. Training *with* thinking enabled would require ~5× longer sequences, pushing activation memory well past 16 GB. Efficient long-CoT RL training (DAPO, Dr. GRPO, FlashAttention-3) is an active research area targeting exactly this gap.